In [9]:
# 導入必要函式庫
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.metrics import confusion_matrix, accuracy_score
import optuna

# 檢查 Kaggle 環境的檔案目錄
import os
print("目錄下的檔案：", os.listdir("/kaggle/input"))

# 載入 Titanic 數據集
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")

# 檢視數據
print("訓練數據預覽：")
print(train_data.head())

# 數據預處理
def preprocess_data(df):
    # 填補缺失值
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    df['Fare'].fillna(df['Fare'].median(), inplace=True)
    
    # 將類別型特徵轉換為數值型
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})
    
    # 選擇有用的特徵
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
    return df[features]

X = preprocess_data(train_data)
y = train_data['Survived']
X_test = preprocess_data(test_data)

# 分割訓練集和驗證集
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# 特徵選擇方法 1: RFE (Recursive Feature Elimination)
def rfe_feature_selection(X, y):
    model = LogisticRegression(max_iter=1000)
    rfe = RFE(model, n_features_to_select=5)  # 選擇 5 個最佳特徵
    rfe.fit(X, y)
    selected_features = X.columns[rfe.support_]
    print("RFE 選擇的特徵：", selected_features)
    return selected_features

# 特徵選擇方法 2: SelectKBest
def selectkbest_feature_selection(X, y):
    kbest = SelectKBest(score_func=f_classif, k=5)  # 選擇 5 個最佳特徵
    kbest.fit(X, y)
    selected_features = X.columns[kbest.get_support()]
    print("SelectKBest 選擇的特徵：", selected_features)
    return selected_features

# 特徵選擇方法 3: Optuna 自動化
def optuna_feature_selection(X, y):
    def objective(trial):
        n_features = trial.suggest_int("n_features", 1, len(X.columns))
        rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=n_features)
        rfe.fit(X, y)
        return rfe.score(X_valid, y_valid)
    
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)
    best_n_features = study.best_params["n_features"]
    print("Optuna 選擇的最佳特徵數量：", best_n_features)
    
    rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=best_n_features)
    rfe.fit(X, y)
    selected_features = X.columns[rfe.support_]
    print("Optuna 選擇的特徵：", selected_features)
    return selected_features

# 呼叫特徵選擇方法
rfe_features = rfe_feature_selection(X_train, y_train)
kbest_features = selectkbest_feature_selection(X_train, y_train)
optuna_features = optuna_feature_selection(X_train, y_train)

# 選擇 Optuna 選出的特徵
X_train_selected = X_train[optuna_features]
X_valid_selected = X_valid[optuna_features]
X_test_selected = X_test[optuna_features]

# 模型訓練
model = LogisticRegression(max_iter=1000)
model.fit(X_train_selected, y_train)

# 預測驗證集
y_pred = model.predict(X_valid_selected)

# 模型評估
conf_matrix = confusion_matrix(y_valid, y_pred)
accuracy = accuracy_score(y_valid, y_pred)

print("\n混淆矩陣：\n", conf_matrix)
print("\n準確率：", accuracy)

# 將預測結果保存為提交文件
y_test_pred = model.predict(X_test_selected)
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": y_test_pred
})
submission.to_csv("submission.csv", index=False)
print("\n提交文件已保存為 submission.csv")


目錄下的檔案： ['titanic']
訓練數據預覽：
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450 

[I 2025-01-13 11:02:09,816] A new study created in memory with name: no-name-eb192e64-9f57-4867-af5d-e43aebabfd4c
[I 2025-01-13 11:02:09,896] Trial 0 finished with value: 0.770949720670391 and parameters: {'n_features': 4}. Best is trial 0 with value: 0.770949720670391.
[I 2025-01-13 11:02:09,946] Trial 1 finished with value: 0.770949720670391 and parameters: {'n_features': 4}. Best is trial 0 with value: 0.770949720670391.
[I 2025-01-13 11:02:09,998] Trial 2 finished with value: 0.770949720670391 and parameters: {'n_features': 4}. Best is trial 0 with value: 0.770949720670391.


SelectKBest 選擇的特徵： Index(['Pclass', 'Sex', 'Parch', 'Fare', 'Embarked'], dtype='object')


[I 2025-01-13 11:02:10,067] Trial 3 finished with value: 0.7821229050279329 and parameters: {'n_features': 1}. Best is trial 3 with value: 0.7821229050279329.
[I 2025-01-13 11:02:10,123] Trial 4 finished with value: 0.770949720670391 and parameters: {'n_features': 4}. Best is trial 3 with value: 0.7821229050279329.
[I 2025-01-13 11:02:10,173] Trial 5 finished with value: 0.770949720670391 and parameters: {'n_features': 4}. Best is trial 3 with value: 0.7821229050279329.
[I 2025-01-13 11:02:10,239] Trial 6 finished with value: 0.7821229050279329 and parameters: {'n_features': 1}. Best is trial 3 with value: 0.7821229050279329.
[I 2025-01-13 11:02:10,297] Trial 7 finished with value: 0.7653631284916201 and parameters: {'n_features': 3}. Best is trial 3 with value: 0.7821229050279329.
[I 2025-01-13 11:02:10,341] Trial 8 finished with value: 0.8100558659217877 and parameters: {'n_features': 6}. Best is trial 8 with value: 0.8100558659217877.
[I 2025-01-13 11:02:10,385] Trial 9 finished wit

Optuna 選擇的最佳特徵數量： 6
Optuna 選擇的特徵： Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Embarked'], dtype='object')

混淆矩陣：
 [[90 15]
 [19 55]]

準確率： 0.8100558659217877

提交文件已保存為 submission.csv
